In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
import pickle
from tqdm import tqdm

In [ ]:
device = "cuda"

In [ ]:
df = pd.read_csv("data/prot_id_content.csv")[:100]



In [ ]:
len(df["content"])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("PharMolix/BioMedGPT-LM-7B")
model = AutoModel.from_pretrained("PharMolix/BioMedGPT-LM-7B").to(device)
model.eval()
emb_dict = {}
for i in tqdm(range(len(df["content"]))):
    encoded_input = tokenizer(df["content"][i], return_tensors="pt")
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    with torch.no_grad():
        output = model(**encoded_input)

    embedding = output.last_hidden_state
    embedding = embedding.squeeze()
    embedding = embedding.mean(dim=0)
    embedding = embedding/torch.linalg.norm(embedding)
    emb_dict[df['id'][i].item()] = embedding.half().cpu()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D", do_lower_case=False)
model = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D").to(device)
model.eval()
emb_dict = {}
for i in tqdm(range(len(df["content"]))):
    encoded_input = tokenizer(df["content"][i], return_tensors="pt")
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    with torch.no_grad():
        output = model(**encoded_input)

    embedding = output.last_hidden_state
    embedding = embedding.squeeze()
    embedding = embedding.mean(dim=0)
    embedding = embedding/torch.linalg.norm(embedding)
    emb_dict[df['id'][i].item()] = embedding.half().cpu()      # x: [D] или [1,D]


In [ ]:
with open("data/dict.pkl", "wb") as f:
    pickle.dump(emb_dict, f)

In [ ]:
# Load model directly


tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-S", trust_remote_code=True)
model = AutoModel.from_pretrained("zhihan1996/DNABERT-S", trust_remote_code=True).to(device)

dna = "CUGCUGCUGCUGCUGCUG"
inputs = tokenizer(dna, return_tensors = 'pt')["input_ids"]
output = model(**encoded_input)

# Эмбеддинги (последний слой)
dna_embeddings = output.last_hidden_state
dna_embeddings = dna_embeddings.squeeze()
dna_embeddings

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("DeepChem/ChemBERTa-100M-MLM")
model = AutoModel.from_pretrained("DeepChem/ChemBERTa-100M-MLM").to(device)
model.eval()
buf_sm = []
for i in df["content"]:
    encoded_input = tokenizer(i, return_tensors="pt")
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    with torch.no_grad():
        output = model(**encoded_input)

    sm_embedding = output.last_hidden_state
    sm_embedding = sm_embedding.squeeze()
    sm_embedding = sm_embedding.mean(dim=0)
    buf_sm.append(sm_embedding.half().cpu())        # x: [D] или [1,D]

T_sm = torch.stack(buf_sm, dim=0)  # итоговый [N,D]